# Z' → tt̄ with GRAEP

End-to-end notebook for the Z' → tt̄ analysis using GRAEP. Each section adds another step to the pipeline.

Right now the notebook covers:

1. Building the fileset.
2. Exporting `nanoaods.json` and the per-process JSONs through the configured `OutputManager`.


In [1]:
import sys
from pathlib import Path

_REPO_ROOT = "/Users/mohamedaly/work/repos/GRAEP"
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from graep.logging import setup_logging  # noqa: E402

setup_logging(3)

## 1. Build the fileset

Pick a resolver, hand it the query callables defined in
`examples/example_opendata_cms/queries.py`. `build_fileset` returns a Python dict; nothing is opened on disk yet.

In [2]:
from examples.example_opendata_cms.config import config  # noqa: E402
from graep.inputs import build_fileset  # noqa: E402

# Defaults to the full set (~555 files, ~15 min cold export). Pass
# `max_files_per_sample=2` while iterating to keep the export fast.
fileset = build_fileset(config.inputs)

print(f"resolved {len(fileset)} processes\n")
for key, entry in fileset.items():
    md = entry["metadata"]
    print(
        f"  {key:30s} files={len(entry['files']):>3}  "
        f"xsec={md['xsec']!s:>8}  is_data={md['is_data']}"
    )


[INFO:graep.inputs.queries:_load_or_compute:L.160] fileset cache hit: /tmp/graep/.cache/d343a1eb382c5b15f59232e4adc56c56.json


resolved 6 processes

  signal__nominal                files=  2  xsec= 0.01895  is_data=False
  ttbar_semilep__nominal         files=138  xsec=  364.31  is_data=False
  ttbar_had__nominal             files=146  xsec=  380.11  is_data=False
  ttbar_lep__nominal             files= 49  xsec=   87.33  is_data=False
  wjets__nominal                 files= 68  xsec= 61526.7  is_data=False
  data__nominal                  files=152  xsec=     1.0  is_data=True


## 2. Export to disk

Build the runtime `OutputManager` from `config.output` and route the export through `mgr["fileset"]`. `export_fileset` opens every file once with uproot, counts entries, sums `genWeight` for MC (skips it for processes flagged `is_data=True`), and writes the JSONs the rest of the pipeline reads.

Cold export of the full set (~555 files) takes around 15 minutes; cap `max_files_per_sample` on the `build_fileset` call above to iterate faster.


In [3]:
import json  # noqa: E402
import time  # noqa: E402

from graep.inputs import export_fileset  # noqa: E402
from graep.output.manager import OutputManager  # noqa: E402

mgr = OutputManager.from_spec(config.output)
fileset_dir = mgr["fileset"]

t0 = time.perf_counter()
export_fileset(fileset, fileset_dir, weights_branch="genWeight")
dt = time.perf_counter() - t0

nanoaods_path = fileset_dir / "nanoaods.json"
per_proc_dir = fileset_dir / "nanoaods_jsons_per_process"
per_proc = sorted(per_proc_dir.glob("*.json")) if per_proc_dir.exists() else []

print(f"export ran in {dt:.1f}s")
print(f"output root:  {mgr.root}")
print(f"combined:     {nanoaods_path}")
print(f"per-process ({len(per_proc)}):")
for p in per_proc:
    print(f"    {p.name}")

print("\ntotals from nanoaods.json:")
with nanoaods_path.open() as f:
    on_disk = json.load(f)
for proc, vars_ in on_disk.items():
    for variation, payload in vars_.items():
        print(
            f"  {proc}__{variation:8s} files={len(payload['files']):>3}  "
            f"nevts_total={payload['nevts_total']:>10}  "
            f"nevts_wt_total={payload['nevts_wt_total']:>14.2f}"
        )


[INFO:graep.inputs.export:export_fileset:L.86] wrote combined fileset JSON: /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/output/datasets/nanoaods.json


[INFO:graep.inputs.export:export_fileset:L.98] wrote per-(process, variation) JSONs under: /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/output/datasets/nanoaods_jsons_per_process


export ran in 820.9s
output root:  /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/output
combined:     /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/output/datasets/nanoaods.json
per-process (6):
    nanoaods_data_nominal.json
    nanoaods_signal_nominal.json
    nanoaods_ttbar_had_nominal.json
    nanoaods_ttbar_lep_nominal.json
    nanoaods_ttbar_semilep_nominal.json
    nanoaods_wjets_nominal.json

totals from nanoaods.json:
  signal__nominal  files=  2  nevts_total=    230000  nevts_wt_total=     230000.00
  ttbar_semilep__nominal  files=138  nevts_total= 144722000  nevts_wt_total=43548252780.00
  ttbar_had__nominal  files=146  nevts_total= 107067000  nevts_wt_total=33608820031.97
  ttbar_lep__nominal  files= 49  nevts_total=  43546000  nevts_wt_total= 3140127367.19
  wjets__nominal  files= 68  nevts_total=  80958227  nevts_wt_total=9697410033920.00
  data__nominal  files=152  nevts_total= 323952013  nevts_wt_total=  323952013.00
